In [ ]:
!pip install "cuml-cu12==25.12.*"

---

In [ ]:
MODELS = [
    "sentence-transformers@sentence-t5-xxl", # No prompt.
    "google@embeddinggemma-300m", # Task-tuned.
    "tencent@KaLM-Embedding-Gemma3-12B-2511" # Instruct-based.
]

In [ ]:
import pandas as pd

In [ ]:
models_dict = {}
for name in MODELS:
    print(name)

    vec_df = pd.read_parquet(f"embeddings_{name}.parquet") # NOTE: The generated embeddings are not uploaded in this anonymous repository due to space limitations.
    models_dict[name] = vec_df

# Datasets

In [ ]:
data_dict = {}

## NRC-VAD

In [ ]:
viz_df = pd.read_csv(
    "opendata/NRC-VAD-Lexicon-v2.1.txt",
    sep='\t'
)

In [ ]:
viz_df = viz_df[~viz_df["term"].isna()]

In [ ]:
viz_df = viz_df.reset_index(drop=True)

In [ ]:
data_dict["NRC-VAD"] = viz_df

## NRC-EIL

In [ ]:
viz_df = pd.read_csv(
    "data/NRC-Emotion-Intensity-Lexicon-v1.txt",
    sep='\t',
    header=None,
    names=["word", "emotion", "intensity"]
)

In [ ]:
viz_df = viz_df.groupby(by="word").agg({
    "emotion": list,
    "intensity": list
})

viz_df = viz_df.rename(columns={"emotion": "emotions", "intensity": "intensities"})

In [ ]:
viz_df = pd.concat(
    [
        viz_df,
        viz_df.apply(lambda x: pd.Series(dict(zip(x["emotions"], x["intensities"]))), axis=1)
    ],
    axis=1
).fillna(0.0)

viz_df = viz_df.drop(columns=["emotions", "intensities"])

In [ ]:
viz_df = viz_df.reset_index()

In [ ]:
data_dict["NRC-EIL"] = viz_df

## GoEmotions

In [ ]:
import io

import requests

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/google-research/google-research/refs/heads/master/goemotions/data"

In [ ]:
emotions = requests.get(f"{DATA_URL}/emotions.txt").text.splitlines()
ekman_mapping = requests.get(f"{DATA_URL}/ekman_mapping.json").json()

inverse_ekman_mapping = {e: k for k, v in ekman_mapping.items() for e in v}

In [ ]:
goemotions = {}

for split in ["train", "dev", "test"]:
    r = requests.get(f"{DATA_URL}/{split}.tsv")

    viz_df = pd.read_csv(
        io.StringIO(r.text),
        sep='\t',
        header=None,
        names=["sentence_text", "category", "id"]
    )

    viz_df["category"] = viz_df["category"].apply(lambda x: [int(e) for e in x.split(',')])
    viz_df["micro_category"] = viz_df["category"].apply(lambda x: [emotions[e] for e in x])
    viz_df["macro_category"] = viz_df["micro_category"].apply(lambda x: [inverse_ekman_mapping.get(e, "neutral") for e in x])

    viz_df = viz_df[viz_df["category"].apply(len) == 1]
    viz_df = viz_df[viz_df["micro_category"].apply(lambda x: "neutral" not in x)]
    for col in ["category", "micro_category", "macro_category"]:
        viz_df[col] = viz_df[col].apply(lambda x: x[0])

    goemotions[split] = viz_df.reset_index(drop=True)

In [ ]:
goemotions = pd.concat([goemotions[split] for split in ["train", "dev", "test"]])

In [ ]:
data_dict["GoEmotions"] = goemotions

# Visualization

## Setup

In [ ]:
from collections import defaultdict

In [ ]:
manifold_dict = defaultdict(dict)

In [ ]:
from cuml.manifold import UMAP

manifold = UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    init="spectral",
    random_state=42
)

In [ ]:
for name in MODELS:
    for key, column in zip(["NRC-VAD", "NRC-EIL", "GoEmotions"], ["term", "word", "sentence_text"]):
        print(name, key)

        X = manifold.fit_transform(
            models_dict[name].loc[data_dict[key][column].astype(str)]
        )
        manifold_dict[name][key] = X.copy().to_dict(orient="list")

In [ ]:
import pickle


with open("viz.pkl", "wb") as f:
    pickle.dump({
        "data_dict": {k: v.to_dict(orient="list") for k, v in data_dict.items()},
        "manifold_dict": manifold_dict
    }, f)